# Starter 2 — Open-source HTR tools: Kraken + PyLaia

> **Status: skeleton.** Structure and interfaces are in place; the model-dependent cells are stubs marked `TODO(kevin)`. Baseline CERs land here before launch.

**Task recap:** given a page image from one of four UW collections (surveyors' field notes, German Kurrent letters, ledger account books, microfilm treaty documents), emit a faithful verbatim transcription. Every model used must be open-weight and under 70B parameters. See the repo README and `docs/transcription_conventions.md`.

Same three-stage pipeline as Starter 1, but each stage is an existing open-source component built for historical documents:

- **[Kraken](https://kraken.re/)** — layout analysis + baseline/line segmentation, designed for messy historical pages
- **[PyLaia](https://gitlab.teklia.com/atr/pylaia)** — CRNN line recognition with published historical-hand models, fine-tunable

This family is what much of the digital-humanities world actually runs in production (it's the engine behind eScriptorium), so it doubles as the most directly library-deployable baseline.

## 0. Setup

Kraken and PyLaia have heavier dependencies than the other starters; a fresh virtualenv is worth it.

In [ ]:
%pip install -q kraken pylaia pandas
# Model zoo: kraken's default segmentation model downloads on first use.
# TODO(kevin): pin exact model versions once baselines are recorded.

## 1. Segmentation with Kraken

Kraken's trainable baseline segmenter handles curved/skewed lines and marks regions — enough to skip plat-map areas of the survey notebooks.

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data/eval")
pages = sorted(DATA_DIR.glob("images/*.jpg"))

# CLI form (equivalent API calls exist):
#   kraken -i page.jpg lines.json segment -bl
# TODO(kevin): run segmentation over sample pages, visualize detected baselines.

## 2. Recognition with PyLaia

PyLaia recognizes each segmented line. Public checkpoints trained on historical corpora exist (search the [Teklia model registry](https://huggingface.co/Teklia) — e.g. models trained on IAM, NewsEye, and other historical sets); fine-tuning on a few hundred in-domain lines is where the score moves.

Routing: use an English-hand model for `survey_notes`/`dominy_accounts`/`treaties_microfilm`, a Kurrent/German model for `kade_letters` (train on Escher if no public checkpoint fits — see `RESOURCES.md`).

In [ ]:
# CLI form:
#   pylaia-htr-decode-ctc --config decode_config.yaml
# TODO(kevin): wire segmented lines -> PyLaia decode -> per-page text assembly,
# with per-category model selection.

## Scoring your output

Whatever pipeline you build, score it the way the official metric does — verbatim CER with whitespace collapsed, macro-averaged by category. `evaluation/metric.py` in the challenge repo is the single source of truth.

In [ ]:
import sys
sys.path.insert(0, "../evaluation")  # adjust if running on Kaggle: attach the challenge repo as a dataset
from metric import page_cer, page_wer

prediction = "North between Sections 13 & 14"
reference  = "North between Sections 13 & 14"
print("CER:", page_cer(prediction, reference), " WER:", page_wer(prediction, reference))

## 3. Fine-tuning notes

Both stages are trainable on modest hardware:

- **Kraken**: `ketos segtrain` on a handful of hand-corrected page segmentations fixes systematic layout misses (tables!)
- **PyLaia**: fine-tune from a historical checkpoint with your curated line pairs; even ~100 lines of survey-notebook tables might yield significant gains on that category

## Ideas to beat this baseline

- eScriptorium as a GUI for producing your fine-tuning data quickly
- Per-collection recognizers instead of per-language
- Confidence-based fallback: send low-confidence lines to the VLM from Starter 3 (both models still under 70B — pipelines may chain models)